# Figure 2e — AT2 vs Club cell identity scores in MHC II High and Low LUAD tumors

**Goal:** Test whether MHC II High and MHC II Low tumor patients differ in their
AT2 and Club cell transcriptional identity scores. The hypothesis is that as
cancer cells become more transformed they lose AT2 identity and downregulate
MHC II expression as part of the same transcriptional shift — making AT2 score
a proxy for the degree of malignant transformation.

**Approach:** Differential gene expression between Club and AT2 cells in the
Salcher atlas is used to derive marker gene sets for each cell type. These
marker scores are computed per cancer cell and aggregated to the sample level.
Samples are then compared between MHC II High and Low groups.

**Input:**
- Salcher et al. atlas (`local.h5ad`) — used for marker gene derivation
- `outputs/processed/luad_mhc2_classified.h5ad` — LUAD object with MHC2_clustering

**Output:** Publication figure 2e — boxplot comparing AT2 and Club scores
between MHC II High and Low patients.

In [ ]:
# standard libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# single-cell analysis
import scanpy as sc
import anndata as ad

# statistics
from scipy.stats import mannwhitneyu

# utilities
from pathlib import Path
import yaml

# figure settings
sns.set(font_scale=1.8)
sns.set_style('ticks')
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42
plt.rcParams.update({'axes.titlesize': 20})

# paper-wide color palettes
# MHC II High = orange (#FF8811), MHC II Low = purple (#462255)
palette_mhc2 = {
    'MHC class II High': '#FF8811',
    'MHC class II Low':  '#462255',
}

In [ ]:
# load paths configuration
with open('/home/gh8sj/projects/mhc2-luad-paper/data/paths_config.yml') as f:
    cfg = yaml.safe_load(f)

repo_root = Path(cfg['repo_root'])

# data paths
atlas_path      = cfg['datasets']['salcher_atlas']['raw']
classified_path = repo_root / cfg['outputs']['processed'] / 'luad_mhc2_classified.h5ad'

# output paths
fig_dir   = repo_root / cfg['outputs']['figures']
table_dir = repo_root / cfg['outputs']['tables']
fig_out   = fig_dir / 'figure2'
table_out = table_dir / 'figure2'
fig_out.mkdir(parents=True, exist_ok=True)
table_out.mkdir(parents=True, exist_ok=True)

# load classified LUAD object
luad = sc.read_h5ad(classified_path)
print(f'Loaded classified LUAD object: {luad.n_obs:,} cells')
print(f'MHC2 classification:\n{luad.obs["MHC2_clustering"].value_counts()}')

# load full atlas for marker gene derivation
# only needed to run rank_genes_groups — skip if markers already defined
adata = sc.read_h5ad(atlas_path)
print(f'Loaded full atlas: {adata.n_obs:,} cells')

## Marker gene derivation

Club and AT2 marker genes are derived by differential expression between the
two cell types in the Salcher atlas using Wilcoxon rank-sum test. The top
markers for each cell type are curated manually from the ranked list to remove
housekeeping genes and retain biologically meaningful markers.

Markers were determined once and are hardcoded below — rerun the
`rank_genes_groups` cells only if the gene list needs to be updated.

In [ ]:
# derive Club markers — top DEGs vs AT2 cells
sc.tl.rank_genes_groups(
    adata,
    groupby='ann_fine',
    groups=['Club'],
    reference='Alveolar cell type 2',
    method='wilcoxon',
    use_raw=False,
)

# inspect top markers
club_candidates = adata.uns['rank_genes_groups']['names']['Club'][:20].tolist()
for i in club_candidates:
    print(adata.var.loc[i]['feature_name'])

In [ ]:
# derive AT2 markers — top DEGs vs Club cells
sc.tl.rank_genes_groups(
    adata,
    groupby='ann_fine',
    groups=['Alveolar cell type 2'],
    reference='Club',
    method='wilcoxon',
    use_raw=False,
)

# inspect top markers
at2_candidates = adata.uns['rank_genes_groups']['names']['Alveolar cell type 2'][:20].tolist()
for i in at2_candidates:
    print(adata.var.loc[i]['feature_name'])

In [ ]:
# curated marker gene sets
# selected from top DEGs — housekeeping genes and non-specific markers removed
#
# Gene sets validated against:
#   Reynolds et al. (2002) Am J Respir Crit Care Med 166:1498–509
#   Whitsett & Weaver (2002) N Engl J Med 347:2141–8
#   Laughney et al. (2020) Nat Med 26:259–69

# Club markers — secretoglobins and mucins specific to Club/secretory cells
club_markers = [
    'WFDC2',   'SCGB1A1', 'SCGB3A1', 'BPIFB1',  'LCN2',
    'CP',      'MDK',     'KLK11',   'TMEM45A',  'TSPAN8',
    'CLU',     'TSPAN1',  'GSTA1',   'SERPINF1', 'TFF3',
    'CXCL1',   'SAA1',    'MUC5B',   'KRT19',    'AGR2',
]

# AT2 markers — surfactant proteins and lipid metabolism genes
at2_markers = [
    'SFTPA1', 'NAPSA',  'SFTPA2', 'SFTPC',  'NPC2',
    'SFTA2',  'SFTPD',  'SLC34A2','CTSH',   'HOPX',
    'FTL',    'PEBP4',  'LRRK2',  'LAMP3',  'LPCAT1',
    'PGC',    'SFTPB',  'DBI',    'ABCA3',  'SFTA3_ENSG00000229415',
]

In [ ]:
# resolve gene symbols to Ensembl IDs
ens_map    = {row['feature_name']: idx for idx, row in luad.var.iterrows()}
club_ids   = [ens_map[g] for g in club_markers  if g in ens_map]
at2_ids    = [ens_map[g] for g in at2_markers   if g in ens_map]

print(f'Club markers resolved:  {len(club_ids)} / {len(club_markers)}')
print(f'AT2 markers resolved:   {len(at2_ids)} / {len(at2_markers)}')

# report any missing genes
missing_club = [g for g in club_markers if g not in ens_map]
missing_at2  = [g for g in at2_markers  if g not in ens_map]
if missing_club:
    print(f'Club markers not found: {missing_club}')
if missing_at2:
    print(f'AT2 markers not found:  {missing_at2}')

## Score computation

Per-cell AT2 and Club scores are computed using Scanpy's `score_genes` function,
which calculates the average expression of the marker gene set relative to a
random control gene set of similar expression level. Scores are then aggregated
to the sample level by taking the mean across all cancer cells per sample.
Only primary tumor cancer cells from MHC II High and Low classified patients
are included — samples labeled 'Excluded from Clustering' are dropped.

In [ ]:
# subset to primary tumor cancer cells with MHC2 classification
tumor_cells = luad[
    (luad.obs['origin'] == 'tumor_primary') &
    (luad.obs['ann_fine'] == 'Cancer cells') &
    (luad.obs['MHC2_clustering'].isin(['MHC class II High', 'MHC class II Low']))
].copy()

print(f'Cancer cells retained: {tumor_cells.n_obs:,}')
print(f'Unique samples: {tumor_cells.obs["sample"].nunique()}')
print(tumor_cells.obs['MHC2_clustering'].value_counts())

In [ ]:
# resolve marker gene symbols to Ensembl IDs
ens_map  = {row['feature_name']: idx for idx, row in tumor_cells.var.iterrows()}
club_ids = [ens_map[g] for g in club_markers if g in ens_map]
at2_ids  = [ens_map[g] for g in at2_markers  if g in ens_map]

print(f'Club markers resolved: {len(club_ids)} / {len(club_markers)}')
print(f'AT2 markers resolved:  {len(at2_ids)} / {len(at2_markers)}')

missing_club = [g for g in club_markers if g not in ens_map]
missing_at2  = [g for g in at2_markers  if g not in ens_map]
if missing_club:
    print(f'Club not found: {missing_club}')
if missing_at2:
    print(f'AT2 not found:  {missing_at2}')

In [ ]:
# compute per-cell scores using scanpy score_genes
# use_raw=False forces scoring on X (log-normalized) rather than raw counts
sc.tl.score_genes(tumor_cells, gene_list=club_ids, score_name='club_score', use_raw=False)
sc.tl.score_genes(tumor_cells, gene_list=at2_ids,  score_name='at2_score',  use_raw=False)

print('Scores computed')
print(tumor_cells.obs[['club_score', 'at2_score']].describe())


In [ ]:
# aggregate to sample-level mean scores
agg_df = (
    tumor_cells.obs
    .groupby(['sample', 'MHC2_clustering'], observed=True)[['club_score', 'at2_score']]
    .mean()
    .reset_index()
)

print(f'Samples aggregated: {len(agg_df)}')
print(agg_df['MHC2_clustering'].value_counts())
agg_df.head()

## Figure 2e — AT2 and Club scores by MHC II classification

MHC II High patients retain higher AT2 identity scores compared to MHC II Low
patients, while Club scores do not differ significantly between groups. This
supports the hypothesis that MHC II downregulation in cancer cells is associated
with loss of AT2 transcriptional identity — as cells become more transformed
they lose AT2 character and downregulate MHC II as part of the same program.

In [ ]:
# reshape for plotting — one row per sample per marker set
plot_scores = pd.melt(
    agg_df,
    id_vars=['sample', 'MHC2_clustering'],
    value_vars=['club_score', 'at2_score'],
    var_name='marker_set',
    value_name='score',
)

# rename for display
plot_scores['marker_set'] = plot_scores['marker_set'].map({
    'club_score': 'Club',
    'at2_score':  'AT2',
})

In [ ]:
sns.set(font_scale=1.2)
sns.set_style('ticks')

fig, ax = plt.subplots(figsize=(6, 6))

HIGH_COLOR = '#FF8811'
LOW_COLOR  = '#462255'

# x positions for 4 groups: Club-High, Club-Low, AT2-High, AT2-Low
positions  = [0, 0.4, 1.2, 1.6]
colors     = [HIGH_COLOR, LOW_COLOR, HIGH_COLOR, LOW_COLOR]
groups     = [
    ('Club', 'MHC class II High'),
    ('Club', 'MHC class II Low'),
    ('AT2',  'MHC class II High'),
    ('AT2',  'MHC class II Low'),
]

data = [
    plot_scores.loc[
        (plot_scores['marker_set'] == marker) &
        (plot_scores['MHC2_clustering'] == cls), 'score'
    ].values
    for marker, cls in groups
]

# draw boxplot
bp = ax.boxplot(
    data,
    positions=positions,
    widths=0.3,
    patch_artist=True,
    medianprops=dict(linewidth=2),
    whiskerprops=dict(linewidth=1),
    capprops=dict(linewidth=1),
    showfliers=False,
)

for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor('none')
    patch.set_edgecolor(color)
    patch.set_linewidth(1.5)

for median, color in zip(bp['medians'], colors):
    median.set_color(color)
    median.set_linewidth(2)

for i, (whisker, color) in enumerate(zip(bp['whiskers'], [c for c in colors for _ in range(2)])):
    whisker.set_color(color)

for i, (cap, color) in enumerate(zip(bp['caps'], [c for c in colors for _ in range(2)])):
    cap.set_color(color)

# overlay scatter points
for vals, x, color in zip(data, positions, colors):
    jitter = np.random.uniform(-0.08, 0.08, size=len(vals))
    ax.scatter(x + jitter, vals, color=color, s=18, alpha=0.7, zorder=3, linewidths=0)

# significance annotations
for i, marker in enumerate(['Club', 'AT2']):
    high = plot_scores.loc[
        (plot_scores['marker_set'] == marker) &
        (plot_scores['MHC2_clustering'] == 'MHC class II High'), 'score'
    ]
    low = plot_scores.loc[
        (plot_scores['marker_set'] == marker) &
        (plot_scores['MHC2_clustering'] == 'MHC class II Low'), 'score'
    ]
    _, p = mannwhitneyu(high, low, alternative='two-sided')
    star = sig_label(p)
    center_x = [0, 1.2][i] + 0.2  # midpoint between High and Low boxes
    ymax = max(np.concatenate([high.values, low.values]))
    yoff = ymax * 0.12
    ax.text(center_x, ymax + yoff, f'{star}\np={p:.2e}',
            ha='center', va='bottom', fontsize=12)

current_ymax = plot_scores['score'].max()
ax.set_ylim(top=current_ymax * 1.35)

# x axis labels
ax.set_xticks([0.2, 1.4])
ax.set_xticklabels(['Club', 'AT2'])
ax.set_xlabel('Marker set')
ax.set_ylabel('Average expression')
ax.set_title('Club vs AT2 marker scores by MHC II clustering')

# manual legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='none', edgecolor=HIGH_COLOR, linewidth=1.5, label='MHC class II High'),
    Patch(facecolor='none', edgecolor=LOW_COLOR,  linewidth=1.5, label='MHC class II Low'),
]
ax.legend(handles=legend_elements, title='MHC II clustering',
          bbox_to_anchor=(1.05, 1), loc='upper left')

ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig(fig_out / 'figure2e_at2_club_scores.pdf', bbox_inches='tight', dpi=300)
plt.show()

In [ ]:
# save summary statistics
stats_rows = []
for marker in ['Club', 'AT2']:
    high = plot_scores.loc[
        (plot_scores['marker_set'] == marker) &
        (plot_scores['MHC2_clustering'] == 'MHC class II High'), 'score'
    ]
    low = plot_scores.loc[
        (plot_scores['marker_set'] == marker) &
        (plot_scores['MHC2_clustering'] == 'MHC class II Low'), 'score'
    ]
    _, p = mannwhitneyu(high, low, alternative='two-sided')
    stats_rows.append({
        'marker_set':        marker,
        'n_high':            len(high),
        'n_low':             len(low),
        'median_high':       float(high.median()),
        'median_low':        float(low.median()),
        'mannwhitney_p':     p,
    })

stats_df = pd.DataFrame(stats_rows)
stats_df.to_csv(table_out / 'figure2e_at2_club_stats.csv', index=False)
print(stats_df)